# Camargo Helpdesk — SharedCat architecture + role-grouped pickles + n-gram loader

Paper-faithful variant. Three changes relative to `train_camargo_LSTM.ipynb`:

1. **Role-grouped pickles** (`helpdesk_all_5_roles_*.pkl`) — `model_feat = [['Activity', 'Role'], ['case_elapsed_time']]`.
2. **SharedCat architecture** — `sharedCatLSTM.model.SharedCat_LSTM` (paper Fig 6b). Table 3: DL act sim 0.9568 on Helpdesk vs 0.5773 for the full-shared variant.
3. **N-gram loader (Camargo §3.1 / Table 1)** — `training.camargo_ngram_dataset.CamargoNGramDataset`. For each case of length L we emit L samples, one per time-step, with a fixed n-gram window of size 5 and the *actual* next activity as target. This replaces the U-ED-LSTM-style encoder-decoder windows (which were EOS-dominated and gave the model no signal at short prefixes).

Target distribution after re-windowing (Helpdesk train): ~20% each of Resolve ticket / EOS / Closed / Assign seriousness / Take in charge ticket. Old loader was 89% EOS.

Checkpoint path differs from the original run so both can be compared side-by-side.

# Imports

In [1]:
import importlib
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
import torch


# Data (role-grouped pickles)

In [2]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
file_path_train = '../../Loader/pkl/helpdesk_all_5_roles_train.pkl'
helpdesk_train_dataset = torch.load(file_path_train, weights_only=False)
print(type(helpdesk_train_dataset))

file_path_val = '../../Loader/pkl/helpdesk_all_5_roles_val.pkl'
helpdesk_val_dataset = torch.load(file_path_val, weights_only=False)
print(type(helpdesk_val_dataset))

<class 'event_log_loader.new_event_log_loader.EventLogDataset'>
<class 'event_log_loader.new_event_log_loader.EventLogDataset'>


In [3]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
helpdesk_all_categories = helpdesk_train_dataset.all_categories
helpdesk_all_categories_cat = helpdesk_all_categories[0]
helpdesk_all_categories_num = helpdesk_all_categories[1]
print(helpdesk_all_categories_cat)
print(helpdesk_all_categories_num)

for i, cat in enumerate(helpdesk_all_categories_cat):
    print(f'Categorical feature: {cat[0]}, idx {i}, #classes={cat[1]}')
for i, num in enumerate(helpdesk_all_categories_num):
    print(f'Numerical feature:   {num[0]}, idx {i}')

concept_name = 'Activity'
concept_name_id = [i for i, cat in enumerate(helpdesk_all_categories_cat) if cat[0] == concept_name][0]
concept_name_size = [cat[1] for cat in helpdesk_all_categories_cat if cat[0] == concept_name][0]
eos_id = [v for k, v in helpdesk_all_categories_cat[concept_name_id][2].items() if k == 'EOS'][0]
print('Activity idx / size / EOS id:', concept_name_id, concept_name_size, eos_id)

[('Activity', 16, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'EOS': 5, 'INVALID': 6, 'Insert ticket': 7, 'RESOLVED': 8, 'Require upgrade': 9, 'Resolve SW anomaly': 10, 'Resolve ticket': 11, 'Schedule intervention': 12, 'Take in charge ticket': 13, 'VERIFIED': 14, 'Wait': 15}), ('Role', 9, {'EOS': 1, 'Role_1': 2, 'Role_2': 3, 'Role_3': 4, 'Role_4': 5, 'Role_5': 6, 'Role_6': 7, 'Role_7': 8})]
[('case_elapsed_time', 1, {})]
Categorical feature: Activity, idx 0, #classes=16
Categorical feature: Role, idx 1, #classes=9
Numerical feature:   case_elapsed_time, idx 0
Activity idx / size / EOS id: 0 16 5


In [4]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
model_feat_cat = [cat[0] for cat in helpdesk_all_categories_cat]
model_feat_num = [num[0] for num in helpdesk_all_categories_num]
model_feat = [model_feat_cat, model_feat_num]
print('model_feat:', model_feat)
assert 'Role' in model_feat_cat, 'Expected the role-grouped pickle. Regenerate it with Helpdesk_full_loader_with_roles.ipynb.'

model_feat: [['Activity', 'Role'], ['case_elapsed_time']]


# Model (SharedCat)

In [5]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
import sharedCatLSTM.model
importlib.reload(sharedCatLSTM.model)
from sharedCatLSTM.model import SharedCat_LSTM

# Same hyper-parameters as the original paper / our reproduction
hidden_size = 50
num_layers = 1
input_size = 1  # sentinel -> SharedCat_LSTM will compute it from embeddings

model = SharedCat_LSTM(
    data_set_categories=helpdesk_all_categories,
    hidden_size=hidden_size,
    num_layers=num_layers,
    model_feat=model_feat,
    input_size=input_size,
    output_size_act=concept_name_size,
)

Data set categories:  ([('Activity', 16, {'Assign seriousness': 1, 'Closed': 2, 'Create SW anomaly': 3, 'DUPLICATE': 4, 'EOS': 5, 'INVALID': 6, 'Insert ticket': 7, 'RESOLVED': 8, 'Require upgrade': 9, 'Resolve SW anomaly': 10, 'Resolve ticket': 11, 'Schedule intervention': 12, 'Take in charge ticket': 13, 'VERIFIED': 14, 'Wait': 15}), ('Role', 9, {'EOS': 1, 'Role_1': 2, 'Role_2': 3, 'Role_3': 4, 'Role_4': 5, 'Role_5': 6, 'Role_6': 7, 'Role_7': 8})], [('case_elapsed_time', 1, {})])
Model input features:  [['Activity', 'Role'], ['case_elapsed_time']]
Embeddings:  ModuleList(
  (0): Embedding(16, 8)
  (1): Embedding(9, 5)
)
Total embedding feature size:  13
Number of numerical features:  1
Input feature size (shared LSTM, cat-only):  13
Cells hidden size:  50
Number of LSTM layer:  1


/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


# Training

In [6]:
import sys
sys.path.insert(0, '../../../../../../..')  # -> src/
import training.camargo_ngram_dataset
import training.train_ngram
importlib.reload(training.camargo_ngram_dataset)
importlib.reload(training.train_ngram)
from training.camargo_ngram_dataset import CamargoNGramDataset
from training.train_ngram import NGramTraining

from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter

# Paper Table 3: n-gram size 5 for Helpdesk.
NGRAM_SIZE = 5

ngram_train = CamargoNGramDataset(helpdesk_train_dataset, ngram_size=NGRAM_SIZE,
                                  activity_idx=concept_name_id, eos_idx=eos_id)
ngram_val = CamargoNGramDataset(helpdesk_val_dataset, ngram_size=NGRAM_SIZE,
                                activity_idx=concept_name_id, eos_idx=eos_id)
print(f'n-gram train: {len(ngram_train)} samples (from {len(helpdesk_train_dataset)} base windows)')
print(f'n-gram val:   {len(ngram_val)} samples (from {len(helpdesk_val_dataset)} base windows)')

writer = SummaryWriter(comment='Full_helpdesk_camargo_sharedcat_roles_ngram5')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# lr=1e-3 (paper-aligned Adam default). The earlier 1e-5 was inherited from the
# U-ED-LSTM setup and is way too small for the Camargo baseline.
learning_rate = 1e-3
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate, weight_decay=0)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=4, min_lr=1e-10)

num_epochs = 100
batch_size = 128
shuffle = True

optimize_values = {
    'optimizer': optimizer,
    'scheduler': scheduler,
    'epochs': num_epochs,
    'mini_batches': batch_size,
    'shuffle': shuffle,
}

trainer = NGramTraining(
    model=model,
    device=device,
    data_train=ngram_train,
    data_val=ngram_val,
    optimize_values=optimize_values,
    writer=writer,
    save_model_n_th_epoch=1,
    saving_path='../pkl/Helpdesk_camargo_sharedcat_roles_ngram5.pkl',
)

trainer.train()

n-gram train: 16895 samples (from 19872 base windows)
n-gram val:   3872 samples (from 4559 base windows)
Device:  cpu
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Scheduler: <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x141fd9a30>
Epochs: 100  Mini-batch: 128  Shuffle: True


  0%|          | 0/100 [00:00<?, ?it/s]

/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch [1/100], LR: 0.001
Training:   Avg Loss: 1.0293
Validation: Avg Loss: 0.4875
saving model
Epoch [2/100], LR: 0.001
Training:   Avg Loss: 0.4739
Validation: Avg Loss: 0.4531
saving model
Epoch [3/100], LR: 0.001
Training:   Avg Loss: 0.4561
Validation: Avg Loss: 0.4459
saving model
Epoch [4/100], LR: 0.001
Training:   Avg Loss: 0.4492
Validation: Avg Loss: 0.4414
saving model
Epoch [5/100], LR: 0.001
Training:   Avg Loss: 0.4447
Validation: Avg Loss: 0.4402
saving model
Epoch [6/100], LR: 0.001
Training:   Avg Loss: 0.4401
Validation: Avg Loss: 0.4365
saving model
Epoch [7/100], LR: 0.001
Training:   Avg Loss: 0.4384
Validation: Avg Loss: 0.4319
saving model
Epoch [8/100], LR: 0.001
Training:   Avg Loss: 0.4364
Validation: Avg Loss: 0.4351
saving model
Epoch [9/100], LR: 0.001
Training:   Avg Loss: 0.4329
Validation: Avg Loss: 0.4317
saving model
Epoch [10/100], LR: 0.001
Training:   Avg Loss: 0.4311
Validation: Avg Loss: 0.4310
saving model
Epoch [11/100], LR: 0.001
Training:   A

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/homebrew/Cellar/python@3.12/3.12.7_1/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.12/3.12.7_1/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/torch/__init__.py", line 2319, in <module>
    from torch import quantization as quantization  # usort: skip
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/philippeichhorn/.local/share/virtualenvs/XAI-Probabilistic_Suffix_Prediction_U-ED-L-p9C7deWr/lib/python3.12/site-packages/torch

KeyboardInterrupt: 